# Lung Cancer Cohort: Data Exploration v11

**Version notes:** v11 applies the 20 May meeting decisions from Hidde and Marieke:

* Age exclusion: cohort restricted to age 18 to 100.
* Duplicate MDN handling: first surgery per patient kept as index event.
* CD outcome rule: NaN CD with `compl = 0` treated as no complication; NaN CD with `compl = 1` or missing left as unknown.
* CD `>=` IIIb framed as the major postoperative complication threshold.
* New `n_known` column in the decision table.
* Markdown note added on CD / LOS correlation.

All v10 structure preserved. Changes are in Section 4.7 (new), Section 7.2 (CD derivation modified), Section 7.7 (new note), Section 9 (decision table modified), and Section 10 (questions updated).

## 0. One time install

`pyreadstat` is usually pre installed on the myDRE VM. The cell below probes for it and only installs (via the myDRE proxy) when the import fails.

In [ ]:
# One-shot install via myDRE proxy. Idempotent — skipped if pyreadstat is
# already importable (which it usually is in the myDRE Anaconda).
import os, sys, subprocess

try:
    import pyreadstat as _pyreadstat_probe
    print(f"pyreadstat already installed (version {_pyreadstat_probe.__version__}); "
          f"skipping install.")
except ImportError:
    os.environ["HTTPS_PROXY"] = "http://proxy.mydre.org:3128"
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        '--only-binary=:all:', '--user', 'pyreadstat'
    ])
    print("pyreadstat installed successfully. You may need to restart the kernel "
          "before the import in Section 1 will pick it up.")

## 1. Setup

Standard Python data science imports plus `pyreadstat` for reading SPSS `.sav` files.

In [ ]:
# Core data manipulation
import pandas as pd
import numpy as np
import re
from pathlib import Path
from collections import Counter

# SPSS .sav reader (loaded in Section 0 install cell)
import pyreadstat

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting style applied to every figure produced in this notebook
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Display options for pandas to surface more information per row by default
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

## 2. Load the Lung Cancer Dataset

The lung cohort is delivered as an SPSS `.sav` file at `Z:\Users\predicting recovery\Lung\DLCA voor Edwin_2.sav`. We read it with `pyreadstat.read_sav`, which returns the data as a `pandas.DataFrame` and the SPSS metadata as a separate `meta` object.

### 2.1 Read the file

In [ ]:
# Project data directory on the myDRE Z drive
lung_sav_path = Path(r"Z:\Users\predicting recovery\Lung\DLCA voor Edwin_2.sav")

print(f"File path: {lung_sav_path}")
print(f"File exists: {lung_sav_path.exists()}")

# pyreadstat returns the DataFrame and the metadata as separate objects.
# user_missing=False (default) maps SPSS-declared missing values to NaN automatically.
lung, meta = pyreadstat.read_sav(str(lung_sav_path), user_missing=False)

print(f"\nShape:                {lung.shape}")
print(f"Rows (patients):      {len(lung):,}")
print(f"Columns (variables):  {lung.shape[1]}")
print(f"\nSPSS file label:      {meta.file_label or '(none)'}")
print(f"SPSS encoding:        {meta.file_encoding}")

### 2.2 Metadata inventory

The `.sav` carries metadata that a CSV export would silently discard. We take stock of what is available before any cleaning: descriptive labels, value label mappings, user defined missing value declarations, and SPSS measurement levels per variable.

In [ ]:
# Counts at the file level
n_vars              = len(meta.column_names)
n_with_var_label    = sum(1 for v in meta.column_names if meta.column_names_to_labels.get(v))
n_with_value_labels = len(meta.variable_value_labels)
n_with_user_missing = (len(meta.missing_user_values) if meta.missing_user_values else 0) \
                    + (len(meta.missing_ranges) if meta.missing_ranges else 0)

print(f"Total variables:                       {n_vars:,}")
print(f"  with descriptive variable labels:    {n_with_var_label:,}")
print(f"  with value-label mappings:           {n_with_value_labels:,}")
print(f"  with user-defined missing values:    {n_with_user_missing:,}")

# Measurement-level distribution (nominal / ordinal / scale)
print(f"\nMeasurement-level distribution:")
print(pd.Series(meta.variable_measure).value_counts().to_string())

# Variable-type breakdown (numeric vs string in original SPSS storage)
print(f"\nOriginal SPSS variable types:")
print(pd.Series(meta.original_variable_types).value_counts().head(10).to_string())

### 2.3 Variable labels for the columns we use most heavily

The DLSA registry uses Dutch short form column names. The `.sav` metadata pairs each column with a descriptive label, which we surface here for the variables that feature in cohort definition, outcomes, and predictors.

In [ ]:
# Show variable labels for the columns we'll use most heavily downstream
key_vars = [
    # Identifiers and dates
    'UPN', 'datok', 'gebdat', 'datont', 'datovl',
    # Cohort and demographics
    'reggroep', 'geslacht', 'Leeftijd', 'lengte', 'gewicht',
    # Fitness and comorbidity
    'asascore', 'ecog', 'roken',
    # Procedure
    'procok', 'toegang', 'typok', 'histtype',
    # Pulmonary function
    'prelongf', 'prelongf1', 'prelongfvo2max',
    # Outcomes
    'compl', 'opnduur', 'status', 'heropname', 'doodoorz',
]

print(f"{'Variable':<22s} {'Label':<70s}")
print('-' * 92)
for v in key_vars:
    if v in meta.column_names:
        label = meta.column_names_to_labels.get(v) or '(no label)'
        print(f"  {v:<20s} {label[:68]:<68s}")
    else:
        print(f"  {v:<20s} (not in dataset)")

### 2.4 DLSA Dictionary cross reference

The DLSA Dictionary (`Datadictionary DLCAS.xls`) carries the registry's authoritative variable definitions, value label declarations, and validation rules. We load it for three purposes: to confirm that the fallback labels for the six drift variables match the dictionary, to extract the mandatory by date rules used in Sections 7.5 and 8.6, and to surface registry level inconsistencies.

If the dictionary is not at the assumed path, this section logs a notice and the dependent sections skip gracefully.

In [ ]:
# v4 fallback mappings for the six variables that v4 had labels for but the
# .sav does not declare value labels for. Cross checked against the DLSA
# Dictionary Opties sheet in Section 2.4.1. Defined here so Section 2.4.1 and
# Section 5.1 can both use it.
V4_FALLBACK_LABEL_MAPS = {
    'ecog': {0: 'ECOG 0 (Normale activiteit)', 1: 'ECOG 1', 2: 'ECOG 2',
             3: 'ECOG 3', 4: 'ECOG 4'},
    'roken': {0: 'Nooit gerookt',
              3: 'Gestopt > 4 weken voor operatie',
              4: 'Gestopt < 4 weken voor operatie',
              5: 'Rookt momenteel nog',
              9: 'Onbekend'},
    'heropname': {0: 'Nee', 1: 'Ja', 9: 'Onbekend'},
    'compluchtweg': {1: 'Luchtlekkage meer dan 5 dagen',
                     2: 'Trachealetsel',
                     3: 'Bronchopleurale fistel (direct ontstaan)',
                     7: 'Andere complicatie van de luchtwegen'},
    'compinfect': {1: 'Bronchopneumonie',
                   2: 'Wondinfectie',
                   3: 'Empyeem (zonder bronchopleurale fistel)',
                   4: 'Empyeem (met bronchopleurale fistel)',
                   5: 'Infectie na mediastinumchirurgie',
                   7: 'Andere infectieuze complicatie'},
    'comprespi': {1: 'meer dan 24 h postoperatief mechanisch beademd',
                  2: 'Longoedeem',
                  3: 'ARDS',
                  4: 'Astma aanval',
                  7: 'Andere respiratoire complicatie'},
}

# Path to the DLSA Dictionary. Adjust if your copy lives elsewhere.
DICT_PATH = Path(r"Z:\Users\predicting recovery\Lung\Datadictionary DLCAS.xls")

dict_loaded = False
dict_vars = None
dict_opties = None
dict_validations = None
mandate_by_date = {}

if not DICT_PATH.exists():
    print(f"Dictionary not found at: {DICT_PATH}")
    print(f"Skipping Section 2.4. Update DICT_PATH if needed.")
else:
    # Variable level sheets
    var_sheets = ['patient', 'comorbiditeiten', 'verrichting', 'palga-pathologie', 'covid19']
    pieces = []
    for s in var_sheets:
        df = pd.read_excel(DICT_PATH, sheet_name=s, dtype=str)
        df['_source_sheet'] = s
        pieces.append(df)
    dict_vars = pd.concat(pieces, ignore_index=True)
    dict_vars = dict_vars[dict_vars['VARIABELE NAAM'].notna()].copy()
    # Strip table prefix like 'verrichting-upn' to plain 'upn'
    dict_vars['var_clean'] = dict_vars['VARIABELE NAAM'].str.replace(
        r'^(patient|verrichting|comorbiditeiten|palga|covid)-', '', regex=True
    )

    dict_opties = pd.read_excel(DICT_PATH, sheet_name='Opties', dtype=str)
    dict_validations = pd.read_excel(DICT_PATH, sheet_name='Validaties', dtype=str)

    print(f"Dictionary loaded:")
    print(f"  variable definitions:    {len(dict_vars):,}")
    print(f"  value label entries:     {len(dict_opties):,}")
    print(f"  validation rules:        {len(dict_validations):,}")

    # Build mandatory by date lookup from the validation expressions
    date_pat = re.compile(r"totime\('(\d{4})-")
    for _, r in dict_validations.iterrows():
        expr = str(r['EXPRESSIE'])
        if '!==null' in expr and 'totime' in expr:
            m = date_pat.search(expr)
            if m:
                v = r['VARIABELE']
                mandate_by_date.setdefault(v, []).append(int(m.group(1)))
    # Reduce to earliest mandatory year per variable
    mandate_by_date = {v: min(years) for v, years in mandate_by_date.items()}
    print(f"  date based mandates:     {len(mandate_by_date):,} variables")

    dict_loaded = True

#### 2.4.1 Confirm the drift variable fallback labels match the dictionary

The six variables for which v6 falls back to a hand maintained dictionary (`ecog`, `roken`, `heropname`, `compinfect`, `compluchtweg`, `comprespi`) get cross checked against the dictionary's Opties sheet. They should match exactly.

In [ ]:
if dict_loaded:
    drift = ['ecog', 'roken', 'heropname', 'compinfect', 'compluchtweg', 'comprespi']
    print(f"Cross check: V4_FALLBACK_LABEL_MAPS vs DLSA Dictionary Opties")
    print(f"{'Variable':<18s} {'Status':<10s} {'Notes'}")
    print('-' * 70)

    all_match = True
    for v in drift:
        codes = dict_opties[dict_opties['VARIABELE'].str.lower() == v.lower()]
        if len(codes) == 0:
            print(f"  {v:<16s} SKIP        not in Opties under this variable name")
            continue
        dict_map = {}
        for _, r in codes.iterrows():
            try:
                dict_map[int(r['WAARDE'])] = r['LABEL']
            except (ValueError, TypeError):
                pass

        our_map = V4_FALLBACK_LABEL_MAPS.get(v, {})

        # Compare keys and content. Tolerate leading numeric prefix and case.
        def normalise(s):
            t = str(s).lower().strip()
            t = re.sub(r'^\d+\.?\s*', '', t)  # leading "0. "
            t = re.sub(r'\s+', ' ', t)
            return t

        missing = set(dict_map) - set(our_map)
        extra = set(our_map) - set(dict_map)
        differs = [k for k in set(dict_map) & set(our_map)
                   if normalise(dict_map[k]) != normalise(our_map[k])]

        if not missing and not extra and not differs:
            print(f"  {v:<16s} OK          {len(dict_map)} codes match")
        else:
            all_match = False
            note_parts = []
            if missing: note_parts.append(f"missing {sorted(missing)}")
            if extra: note_parts.append(f"extra {sorted(extra)}")
            if differs: note_parts.append(f"differs on {sorted(differs)}")
            print(f"  {v:<16s} DIFFERS     {', '.join(note_parts)}")

    if all_match:
        print(f"\nResult: all six fallback maps match the DLSA Dictionary exactly.")

#### 2.4.2 Mandatory by date rules from the Validaties sheet

The registry phases in new mandatory fields over time. Without knowing these dates, the coverage percentages reported in Sections 7 and 8 are misleading. The table below summarises the date based mandate for the variables this notebook uses.

In [ ]:
if dict_loaded:
    of_interest = [
        'lengte', 'gewicht', 'ecog', 'asascore',
        'prelongf', 'prelongf1', 'prelongfvo2max',
        'clavalg', 'clavrespi', 'clavinfect', 'clavluchtweg', 'clavmechan',
        'compl', 'compover', 'compinfect', 'compluchtweg', 'comprespi',
        'compcardio', 'compproc', 'comptrombo',
        'roken', 'opnduur', 'heropname', 'doodoorz', 'status',
    ]
    print(f"Mandatory by date rules for variables this notebook uses:")
    print(f"{'Variable':<20s} {'Mandatory from'}")
    print('-' * 40)
    for v in of_interest:
        y = mandate_by_date.get(v)
        if y:
            print(f"  {v:<18s} {y}")
        else:
            print(f"  {v:<18s} no date based mandate")

#### 2.4.3 Inventory AUMC specific columns not in the DLSA Dictionary

Columns may exist in the `.sav` that the DLSA Dictionary does not document. These are typically AUMC additions to the registry export, such as locally tracked outcome variables or derived fields. We surface them here so they are not invisible to the rest of the notebook.

In [ ]:
if dict_loaded:
    dict_vars_set = set(dict_vars['var_clean'].str.lower())
    sav_vars_lower = {c.lower(): c for c in lung.columns}

    extra = sorted(name for low, name in sav_vars_lower.items() if low not in dict_vars_set)
    # Drop our own derived columns (UPN_raw, _label, _dt, etc) to keep the list focused
    derived_suffixes = ('_raw', '_clean', '_length', '_label', '_dt', '_local')
    derived_names = {'BMI', 'CD_max', 'CD_max_label', 'CD_ge_II', 'CD_ge_IIIa',
                     'CD_ge_IIIb', 'LOS_prolonged', 'mortality_30d',
                     'n_comorbidities', 'surgery_year'}
    extra = [v for v in extra
             if not v.endswith(derived_suffixes) and v not in derived_names]

    print(f"Columns present in the .sav but not in the DLSA Dictionary: {len(extra)}")
    for v in extra:
        dtype = lung[v].dtype
        n = lung[v].notna().sum()
        print(f"  {v:<25s} dtype={str(dtype):<12s} non-null={n:,}")
else:
    print("Dictionary not loaded. Skipping AUMC specific column inventory.")

## 3. Normalise the UPN Identifier

The UPN column should contain a 7 digit code uniquely identifying each patient. Six digit values get a leading zero padded back on; anything else is flagged and exported as an outlier for clinical review. The Edwin notes dataset uses the same identifier under the column name MDN, so this normalisation is the key the downstream linkage notebook joins on.

In the `.sav` the UPN is stored numerically. We convert to integer string first so leading zero behaviour stays consistent across rows.

### 3.1 Inspect raw UPN length distribution

In [ ]:
# UPN comes from the .sav as a numeric column for most rows, but a handful of
# values arrive as scientific-notation strings (e.g. '8.5092071E7'). Convert
# through float() first so those strings are normalised to their integer form.
def upn_to_str(v):
    if pd.isna(v):
        return None
    try:
        # float() handles both numerics and scientific-notation strings
        f = float(v)
        return str(int(f))
    except (ValueError, TypeError):
        return str(v).strip()

lung['UPN'] = lung['UPN'].apply(upn_to_str)

print("UPN before normalisation:")
print(f"  dtype:                {lung['UPN'].dtype}")
print(f"  non null count:       {lung['UPN'].notna().sum():,} / {len(lung):,}")
print(f"  length distribution (digits):")
print(lung['UPN'].dropna().str.len().value_counts().sort_index().to_string())

### 3.2 Identify and export UPN outliers

In [ ]:
# Compute UPN length and flag outliers
lung['UPN_clean'] = lung['UPN'].astype(str).str.strip()
lung['UPN_length'] = lung['UPN_clean'].str.len()

# Outliers: any UPN length other than 6 or 7
outlier_mask = ~lung['UPN_length'].isin([6, 7])
outliers = lung.loc[outlier_mask].copy()

print(f"UPN outlier rows (length not 6 or 7): {len(outliers):,}")
if len(outliers) > 0:
    print(f"\nOutlier UPN values and their lengths:")
    print(outliers[['UPN_clean', 'UPN_length']].to_string(index=False))

# Export outliers to a dedicated review file
output_dir = Path(r"Z:\Users\predicting recovery\AmanDeep\Data Analysis\lung_analysis\output_files")
output_dir.mkdir(parents=True, exist_ok=True)

outlier_path = output_dir / "lung_upn_outliers.csv"
outliers.to_csv(outlier_path, index=False, encoding='utf-8')
print(f"\nOutliers exported to: {outlier_path}")
print(f"  Rows: {len(outliers):,}")

### 3.3 Apply normalisation to legitimate UPN values

In [ ]:
# Preserve the original UPN for audit
lung['UPN_raw'] = lung['UPN']

# Normalise: pad 6 digit values to 7; leave others as is
def normalise_upn(val):
    if not isinstance(val, str):
        return val
    v = val.strip()
    if len(v) == 6:
        return '0' + v
    return v

lung['UPN'] = lung['UPN_clean'].apply(normalise_upn)

# Verify the result
print("UPN after normalisation:")
print(f"  length distribution (digits):")
print(lung['UPN'].dropna().str.len().value_counts().sort_index().to_string())

n_padded = (lung['UPN_raw'].astype(str).str.strip().str.len() == 6).sum()
print(f"\nRows padded from 6 to 7 digits: {n_padded:,}")
print(f"Outlier UPN rows retained (not 6 or 7 digit): {(lung['UPN'].str.len() != 7).sum():,}")

n_total = len(lung)
n_unique = lung['UPN'].nunique()
print(f"\nUnique UPN values: {n_unique:,} (of {n_total:,} rows)")
if n_unique < n_total:
    print(f"  Note: {n_total - n_unique:,} duplicate UPN rows exist, "
          f"this is expected if a patient had multiple surgeries.")


### 3.4 Inspect duplicate UPN rows

Eight UPN values appear on more than one row. This is legitimate when a patient has multiple thoracic procedures at different points in time. We surface the duplicates here to confirm that interpretation and expose any rows that are not legitimately the same patient on a different surgery date.

In [ ]:
dup_upns = lung.loc[
    lung['UPN'].duplicated(keep=False) & lung['UPN'].notna(),
    'UPN'
].unique()

print(f"Distinct UPN values appearing on more than one row: {len(dup_upns)}")

if len(dup_upns) > 0:
    keep_cols = ['UPN']
    for c in ['datok', 'datok_dt', 'reggroep', 'procok']:
        if c in lung.columns:
            keep_cols.append(c)
    # Add label cols if they exist
    for c in ['reggroep_label', 'procok_label']:
        if c in lung.columns:
            keep_cols.append(c)

    dup_view = lung.loc[lung['UPN'].isin(dup_upns), keep_cols].copy()
    if 'datok_dt' in dup_view.columns:
        dup_view = dup_view.sort_values(['UPN', 'datok_dt'])
    else:
        dup_view = dup_view.sort_values(['UPN'])

    print(f"\nDuplicate UPN inspection:")
    print(dup_view.to_string(index=False))

    dup_path = output_dir / "lung_upn_duplicates.csv"
    dup_view.to_csv(dup_path, index=False, encoding='utf-8')
    print(f"\nExported to: {dup_path}")

## 4. Sentinel Verification and ISO Date Handling

The DLSA registry uses specific numeric codes for missing values: `9 = Onbekend` for binary variables and Clavien Dindo grades, `999` (or `99.9`) for pulmonary function measurements. The `.sav` metadata does not declare these as user defined missing, so `pyreadstat` does not auto recode them. This section applies the manual recoding using lists transcribed from the DLSA Dictionary.

### 4.1 Recode the 9 = Onbekend sentinel in binary variables

35 binary variables follow the standard `0=Nee / 1=Ja / 9=Onbekend` pattern (DLSA option set 501). We check the `.sav` user missing declarations, report what `pyreadstat` already handled, and apply manual recoding for any 9 values still present.

In [ ]:
# v4's hand-curated list of binary variables with the 9=Onbekend sentinel
BINARY_VARS_WITH_9_UNKNOWN = [
    'bronresec', 'carinresec', 'chylo', 'compl', 'compover', 'compreint',
    'ctnmbekend', 'ctscan', 'ebus', 'eus', 'heropname', 'koolhydraten',
    'letsel', 'mediasti', 'opiaat-dag3', 'pet', 'postopmdo',
    'tumorpetpos', 'preopmdt', 'transfusie',
    'invasie', 'invasbron', 'invaspleura', 'invasthw', 'invasphren',
    'invasperic', 'invasdiafr', 'invasmediast', 'invashart', 'invasvaat',
    'invastrach', 'invasrecur', 'invasoesof', 'invaswervel', 'invascarina',
]

print(f"v4 list size: {len(BINARY_VARS_WITH_9_UNKNOWN)} variables expecting 9=Onbekend\n")

# Check what the .sav metadata declares as user_missing for each, and how
# many 9 values remain in the data after auto-recoding.
still_has_9 = []
header = f"{'Variable':<18s} {'sav user_missing':<22s} {'remaining 9 in data':>22s}"
print(header)
print('-' * len(header))
for col in BINARY_VARS_WITH_9_UNKNOWN:
    if col not in lung.columns:
        continue
    declared = meta.missing_user_values.get(col, [])
    n_9 = int((lung[col] == 9).sum())
    print(f"  {col:<16s} {str(declared):<22s} {n_9:>22,d}")
    if n_9 > 0:
        still_has_9.append((col, n_9))

# Apply manual recoding only where the .sav metadata missed it
if still_has_9:
    print(f"\n{len(still_has_9)} variables still contain 9s — applying manual recoding "
          f"(9 -> NaN)")
    total = 0
    for col, n in still_has_9:
        lung[col] = lung[col].replace(9, np.nan)
        total += n
    print(f"Total cells recoded: {total:,}")
else:
    print(f"\nAll v4 sentinels already handled by .sav metadata — no manual recoding needed.")

### 4.2 Recode the 999 = Onbekend sentinel in numeric measurements

For continuous measurements like pulmonary function (FEV1, VO2 max), the DLSA registry uses code `999` to indicate Onbekend (`99.9` for VO2). Some columns are stored as A typed strings in the `.sav`, so we coerce them with `pd.to_numeric` first, then apply recoding.

In [ ]:
# Pulmonary function variables that v4 recoded manually
PULM_VARS_WITH_999 = ['prelongf', 'prelongf1', 'prelongfvo2max']

# Some columns in the .sav are stored as A10 strings even though they encode
# numeric values (prelongfvo2max in particular). Coerce to numeric FIRST so
# diagnostics and arithmetic work.
print("Coercing pulmonary columns to numeric (handles A-typed string storage):")
for col in PULM_VARS_WITH_999:
    if col in lung.columns:
        before_dtype = lung[col].dtype
        lung[col] = pd.to_numeric(lung[col], errors='coerce')
        print(f"  {col:<20s} {str(before_dtype):<10s} -> {str(lung[col].dtype)}")

# What does the .sav say about user-missing for these variables?
print("\nMetadata for pulmonary function variables:")
for col in PULM_VARS_WITH_999:
    if col in lung.columns:
        declared = meta.missing_user_values.get(col, [])
        ranges = meta.missing_ranges.get(col, []) if meta.missing_ranges else []
        print(f"  {col:<20s} user_missing={declared}, missing_ranges={ranges}")

# Diagnostic: what sentinels are present before recoding?
print(f"\nBefore additional recoding:")
for col in PULM_VARS_WITH_999:
    if col in lung.columns:
        n_999 = int((lung[col] == 999).sum())
        n_99 = int((lung[col] == 99.9).sum())
        mean = lung[col].mean()
        mean_str = f"{mean:.1f}" if pd.notna(mean) else "(all null)"
        print(f"  {col:<20s} 999 count: {n_999:,}, 99.9 count: {n_99:,}, "
              f"mean before: {mean_str}")

# Apply manual recoding
print(f"\nApplying manual recoding (999 and 99.9 -> NaN):")
for col in PULM_VARS_WITH_999:
    if col in lung.columns:
        before = lung[col].copy()
        lung[col] = lung[col].replace([999, 99.9], np.nan)
        n_changed = int((before.notna() & lung[col].isna()).sum())
        if n_changed > 0:
            print(f"  {col:<20s} recoded {n_changed:,} cells")

print(f"\nAfter additional recoding:")
for col in PULM_VARS_WITH_999:
    if col in lung.columns:
        n = int(lung[col].notna().sum())
        if n > 0:
            print(f"  {col:<20s} non null: {n:,} ({n/len(lung)*100:.1f}%), "
                  f"mean: {lung[col].mean():.1f}, "
                  f"range: [{lung[col].min():.1f}, {lung[col].max():.1f}]")
        else:
            print(f"  {col:<20s} non null: 0")

### 4.3 Recode the 9 = Onbekend sentinel in Clavien Dindo grade columns

The Clavien Dindo grade columns (`clav*`) use code 9 = Onbekend. We apply manual recoding for any 9 values present.

In [ ]:
# Clavien-Dindo grade columns use code 9 = Onbekend
CLAV_COLS = [c for c in lung.columns if c.startswith('clav') and not c.endswith('_label')]

print(f"Clavien-Dindo grade columns found: {len(CLAV_COLS)}")

# How many have user-missing declared in the .sav?
n_with_missing_declared = sum(
    1 for c in CLAV_COLS
    if c in meta.missing_user_values and 9 in meta.missing_user_values[c]
)
print(f"  with 9 declared as user-missing in .sav: {n_with_missing_declared}")

# Apply manual recoding only where 9s remain
n_clav_9_total = 0
for col in CLAV_COLS:
    n_9 = int((lung[col] == 9).sum())
    if n_9 > 0:
        lung[col] = lung[col].replace(9, np.nan)
        n_clav_9_total += n_9

if n_clav_9_total > 0:
    print(f"  manually recoded 9 -> NaN: {n_clav_9_total:,} cells")
else:
    print(f"  no leftover 9s — .sav metadata handled all CD sentinels")

### 4.4 Confirm the SPSS date columns parsed as datetime

`pyreadstat` converts SPSS date and datetime columns to pandas `datetime64[ns]` automatically. We confirm the four date columns are already in datetime format and alias them with the `_dt` suffix used by downstream code.

In [ ]:
# Confirm the four date columns came in as datetime and alias them to <col>_dt
ISO_DATE_COLS = ['datok', 'gebdat', 'datont', 'datovl']

for col in ISO_DATE_COLS:
    if col not in lung.columns:
        print(f"  {col:<12s} (not in dataset)")
        continue
    dtype_now = str(lung[col].dtype)
    if 'datetime' in dtype_now:
        lung[col + '_dt'] = lung[col]
        s = lung[col + '_dt']
        n = int(s.notna().sum())
        print(f"  {col:<12s} -> {col + '_dt':<15s} {n:,} valid, "
              f"range: {s.min().date()} to {s.max().date()} (already datetime)")
    else:
        # Defensive fallback for columns SPSS may have stored as string
        lung[col + '_dt'] = pd.to_datetime(lung[col], errors='coerce')
        n = int(lung[col + '_dt'].notna().sum())
        print(f"  {col:<12s} -> {col + '_dt':<15s} {n:,} valid (parsed with pd.to_datetime)")

### 4.5 Derive the surgery year

With `datok_dt` in datetime format, we extract the surgery year for temporal analysis.

In [ ]:
# Derive surgery year for temporal analysis
if 'datok_dt' in lung.columns:
    lung['surgery_year'] = lung['datok_dt'].dt.year
    print(f"Surgery year distribution:")
    print(lung['surgery_year'].value_counts().sort_index().to_string())


### 4.6 Cross field date sanity check

The DLSA Dictionary's Validaties sheet specifies four cross field date rules. `datok` must be after `gebdat`. `datont` must be at or after `datok`. `datovl` must be at or after `gebdat` and not in the future. If `status` equals 1 (deceased), `datovl` is required. We check each rule and report violations.

In [ ]:
print("Cross field date sanity checks (rules from DLSA Dictionary Validaties):")

def report(name, mask):
    n = int(mask.sum())
    flag = "OK" if n == 0 else "VIOLATION"
    print(f"  {name:<45s} {flag} ({n} rows)")
    return n

# Rule 1: datok must be after gebdat
mask1 = (lung['datok_dt'].notna()
         & lung['gebdat_dt'].notna()
         & (lung['datok_dt'] <= lung['gebdat_dt']))
report("datok strictly after gebdat", mask1)

# Rule 2: datont must be at or after datok
mask2 = (lung['datont_dt'].notna()
         & lung['datok_dt'].notna()
         & (lung['datont_dt'] < lung['datok_dt']))
report("datont at or after datok", mask2)

# Rule 3a: datovl must be at or after gebdat
mask3a = (lung['datovl_dt'].notna()
          & lung['gebdat_dt'].notna()
          & (lung['datovl_dt'] < lung['gebdat_dt']))
report("datovl at or after gebdat", mask3a)

# Rule 3b: datovl not in the future
today = pd.Timestamp.today().normalize()
mask3b = lung['datovl_dt'].notna() & (lung['datovl_dt'] > today)
report("datovl not in the future", mask3b)

# Rule 4: status = 1 (Overleden) implies datovl is populated
status1 = (lung['status'] == 1)
mask4 = status1 & lung['datovl_dt'].isna()
n4 = report("status=Overleden implies datovl present", mask4)

if n4 > 0:
    # At this point in the notebook, _label columns from Section 5 do not yet
    # exist. Use the numeric status column, add status_label only if available.
    keep = ['UPN', 'datok_dt', 'status']
    if 'status_label' in lung.columns:
        keep.append('status_label')
    affected = lung.loc[mask4, keep].copy()
    print(f"\n  Affected rows (status=Overleden with datovl missing):")
    print(affected.to_string(index=False))

### 4.7 Apply Cohort Exclusions Per 20 May Meeting Decisions

Hidde and Marieke agreed on the following data quality exclusions at the 20 May meeting:

**Age exclusion.** Exclude patients with age at surgery outside the 18 to 100 range. Children under 18 are not part of the surgical lung cancer population. Ages above 100 are data entry errors (the surgical team enters today's date as a placeholder when the original date is unavailable in EPIC). The 3 violations from Section 4.6 (`datok <= gebdat`) will be caught by the age < 18 rule.

**Duplicate MDN handling.** For each patient appearing in multiple rows, keep the first surgery (earliest `datok_dt`) as the index event. Recurrence or reoperation cases shift the risk profile and would bias the analysis if treated as a fresh patient.

Both exclusions are applied here, before downstream cohort definition.

In [ ]:
# 4.7.1 Compute age at surgery and apply the 18 to 100 exclusion
lung['age_at_surgery'] = ((lung['datok_dt'] - lung['gebdat_dt']).dt.days / 365.25).round(2)

n_before = len(lung)
mask_age_invalid = (
    lung['age_at_surgery'].notna()
    & ((lung['age_at_surgery'] < 18) | (lung['age_at_surgery'] > 100))
)
n_age_violations = int(mask_age_invalid.sum())

print(f"Age at surgery distribution before exclusion:")
print(f"  n with computable age:  {int(lung['age_at_surgery'].notna().sum()):,}")
print(f"  min:                    {lung['age_at_surgery'].min():.1f}")
print(f"  median:                 {lung['age_at_surgery'].median():.1f}")
print(f"  max:                    {lung['age_at_surgery'].max():.1f}")

print(f"\nAge outliers (< 18 or > 100): {n_age_violations}")
if n_age_violations > 0:
    print(lung.loc[mask_age_invalid, ['UPN', 'datok_dt', 'gebdat_dt', 'age_at_surgery']]
          .to_string(index=False))

# Apply the exclusion
lung = lung.loc[~mask_age_invalid].copy()
print(f"\nRows after age exclusion: {len(lung):,} (dropped {n_before - len(lung):,})")

In [ ]:
# 4.7.2 Deduplicate to one row per UPN, keeping the earliest surgery as the index event
n_before = len(lung)
n_unique_upn_before = int(lung['UPN'].nunique())

lung = (lung.sort_values('datok_dt')
            .drop_duplicates(subset='UPN', keep='first')
            .copy())

n_dropped = n_before - len(lung)
print(f"Duplicate handling per Hidde's rule (keep first surgery per UPN):")
print(f"  Rows before dedup:           {n_before:,}")
print(f"  Unique UPN before dedup:     {n_unique_upn_before:,}")
print(f"  Rows after dedup:            {len(lung):,}")
print(f"  Rows dropped:                {n_dropped}")
print(f"  Unique UPN after dedup:      {int(lung['UPN'].nunique()):,}")

## 5. Apply Value Labels from the `.sav` Metadata

The DLSA registry stores categorical variables as numeric codes. The mappings come from the `.sav` metadata directly via `meta.variable_value_labels`. For the six variables the `.sav` omits but the dictionary documents (`ecog`, `roken`, `heropname`, `compluchtweg`, `compinfect`, `comprespi`), we fall back to a small hand maintained dictionary.

### 5.1 Source mappings from the metadata and apply fallback for drift variables

In [ ]:
# v4 hand-curated list, retained as a cross-check
V4_LABEL_VARS = {
    'roken', 'geslacht', 'asascore', 'ecog', 'reggroep', 'toegang', 'typok',
    'procok', 'histtype', 'status', 'doodoorz', 'radicali', 'sleeve',
    'compluchtweg', 'compinfect', 'comprespi', 'compl', 'heropname',
}

# V4_FALLBACK_LABEL_MAPS is defined in Section 2.4; reuse it here.

# v5: source value-label mappings primarily from the .sav metadata, fall back
# to V4_FALLBACK_LABEL_MAPS for the drift variables.
LABEL_MAPS = dict(meta.variable_value_labels)  # copy so we can extend safely
n_from_sav = len(LABEL_MAPS)

for var, mapping in V4_FALLBACK_LABEL_MAPS.items():
    if var not in LABEL_MAPS and var in lung.columns:
        LABEL_MAPS[var] = mapping

n_from_fallback = len(LABEL_MAPS) - n_from_sav

print(f"Value-label mappings sourced:")
print(f"  from .sav metadata:  {n_from_sav:,}")
print(f"  from v4 fallback:    {n_from_fallback}")
print(f"  total available:     {len(LABEL_MAPS):,}")

# Sample preview
print(f"\nSample mappings:")
for var in ['geslacht', 'reggroep', 'asascore', 'ecog', 'roken']:
    if var in LABEL_MAPS:
        src = 'sav' if var in meta.variable_value_labels else 'v4 fallback'
        print(f"\n  {var} [{src}]:")
        for k, v in LABEL_MAPS[var].items():
            print(f"    {k} -> {v}")

# Cross-check against v4's curated list
sav_keys = set(meta.variable_value_labels)
in_sav_only = sav_keys - V4_LABEL_VARS
in_v4_only  = V4_LABEL_VARS - sav_keys

print(f"\nCross-check vs v4 hand-curated list ({len(V4_LABEL_VARS)} variables):")
print(f"  in .sav but not v4:  {len(in_sav_only)} additional variables now labelled")
print(f"  in v4 but not .sav:  {len(in_v4_only)} drift candidates")
if in_v4_only:
    print(f"    drift candidates:  {sorted(in_v4_only)}")
    print(f"    -> using V4_FALLBACK_LABEL_MAPS for {len(V4_FALLBACK_LABEL_MAPS)} of these")

# Clavien-Dindo grade mapping
clav_with_labels = [c for c in CLAV_COLS if c in LABEL_MAPS]
if clav_with_labels:
    CD_GRADE_MAP = LABEL_MAPS[clav_with_labels[0]]
    print(f"\nCD grade mapping sourced from .sav ({clav_with_labels[0]}, "
          f"{len(CD_GRADE_MAP)} grades).")
else:
    CD_GRADE_MAP = {
        1: 'Graad I', 2: 'Graad II', 3: 'Graad IIIa', 4: 'Graad IIIb',
        5: 'Graad IVa', 6: 'Graad IVb', 7: 'Graad V (overleden)',
    }
    print(f"\nCD grade mapping NOT in .sav metadata — using v4 manual fallback.")

### 5.2 Apply the value labels

In [ ]:
# Apply label mappings, creating new <col>_label columns
n_columns_labelled = 0
for col, mapping in LABEL_MAPS.items():
    if col in lung.columns:
        lung[col + '_label'] = lung[col].map(mapping)
        n_columns_labelled += 1

# Make sure every clav* column has a _label sibling, falling back to CD_GRADE_MAP
# for any that the metadata-driven loop above didn't cover.
for col in CLAV_COLS:
    label_col = col + '_label'
    if label_col not in lung.columns:
        lung[label_col] = lung[col].map(CD_GRADE_MAP)
        n_columns_labelled += 1

print(f"Created {n_columns_labelled} labelled columns (suffix _label).")

### 5.3 Verify labelled output on key variables

In [ ]:
# Quick spot check on the most important variables
print("=== Sex (geslacht) ===")
print(lung['geslacht_label'].value_counts(dropna=False).to_string())

print("\n=== ASA score (asascore) ===")
print(lung['asascore_label'].value_counts(dropna=False).to_string())

print("\n=== ECOG performance status (ecog) ===")
print(lung['ecog_label'].value_counts(dropna=False).to_string())

print("\n=== Smoking status (roken) ===")
print(lung['roken_label'].value_counts(dropna=False).to_string())

print("\n=== Procedure performed (procok) ===")
print(lung['procok_label'].value_counts(dropna=False).to_string())

print("\n=== Surgical approach (toegang) ===")
print(lung['toegang_label'].value_counts(dropna=False).to_string())

print("\n=== Histological type (histtype) ===")
print(lung['histtype_label'].value_counts(dropna=False).to_string())


## 6. Cohort Definition by Registration Group

The DLSA captures all thoracic surgery performed at AUMC. Patients are categorised by `reggroep`: 1 = Resectie longcarcinoom (primary lung cancer resection), 2 = Mediastinumchirurgie, 3 = Metastasectomie, 4 = Overige ingreep. The analytical cohort decision depends on whether we restrict to primary lung cancer only, include metastasectomy, or use all registrants.

### 6.1 Registration group breakdown

In [ ]:
# Cohort composition by registration group
print("Registration group distribution:")
print(lung['reggroep_label'].value_counts(dropna=False).to_string())

print(f"\nTotal rows in raw export:  {len(lung):,}")
print(f"Total with reggroep populated: {lung['reggroep'].notna().sum():,}")


In [ ]:
# Bar chart for visual reference
fig, ax = plt.subplots(figsize=(10, 5))
counts = lung['reggroep_label'].value_counts()
counts.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title("Registration group composition of the raw lung export (n = "
             f"{lung['reggroep'].notna().sum():,})")
ax.set_xlabel("Number of records")
for i, v in enumerate(counts.values):
    ax.text(v + 5, i, f"{v:,}", va='center')
plt.tight_layout()
plt.show()


### 6.2 Define the candidate analytical cohorts

We create three cohort views without dropping anything from the DataFrame. The choice between them is made downstream after Edwin reviews the decision table.

In [ ]:
# Three candidate analytical cohorts
cohort_A = lung['reggroep'] == 1                          # primary lung cancer only
cohort_B = lung['reggroep'].isin([1, 3])                  # primary lung cancer + metastasectomy
cohort_C = lung['reggroep'].isin([1, 2, 3, 4])            # all registered cases

print(f"Cohort A (Resectie longcarcinoom only):                       {cohort_A.sum():,}")
print(f"Cohort B (Lung resection + Metastasectomy):                   {cohort_B.sum():,}")
print(f"Cohort C (All thoracic surgery, all registration groups):     {cohort_C.sum():,}")
print(f"\nUnique UPN counts per cohort:")
print(f"  Cohort A: {lung.loc[cohort_A, 'UPN'].nunique():,}")
print(f"  Cohort B: {lung.loc[cohort_B, 'UPN'].nunique():,}")
print(f"  Cohort C: {lung.loc[cohort_C, 'UPN'].nunique():,}")


## 7. Outcome Variable Inventory

Identifies the candidate outcomes and reports coverage and event rate within each candidate cohort.

### 7.1 Any complication

In [ ]:
print("compl (Any complication, after 9->NaN recoding):")
print(f"  Coverage:    {lung['compl'].notna().sum():,} / {len(lung):,} ({lung['compl'].notna().mean()*100:.1f}%)")
print(f"  Value counts:")
print(lung['compl_label'].value_counts(dropna=False).to_string())

print(f"\nEvent rate (compl = Ja) within each candidate cohort:")
for name, mask in [('A (Resectie longcarcinoom)', cohort_A),
                   ('B (Resectie + Metastasectomie)', cohort_B),
                   ('C (All thoracic surgery)', cohort_C)]:
    n_total = mask.sum()
    n_event = ((mask) & (lung['compl'] == 1)).sum()
    rate = n_event / n_total * 100 if n_total else 0
    print(f"  Cohort {name:<35s} {n_event:>4,d}/{n_total:>5,d} = {rate:>5.1f}%")


### 7.2 Patient level maximum Clavien Dindo grade

We compute `CD_max` as the highest Clavien Dindo grade across the patient's recorded complications, then derive binary event indicators at the three standard severity thresholds.

In [ ]:
# Patient level maximum Clavien Dindo grade
lung['CD_max'] = lung[CLAV_COLS].max(axis=1)
lung['CD_max_label'] = lung['CD_max'].map(CD_GRADE_MAP)

print(f"Patient level maximum Clavien Dindo grade (CD_max):")
print(f"  non null: {lung['CD_max'].notna().sum():,} ({lung['CD_max'].notna().mean()*100:.1f}%)")
print(f"  distribution:")
print(lung['CD_max_label'].value_counts(dropna=False).sort_index().to_string())

# ----------------------------------------------------------------------------
# Derived event indicators under Hidde's rule (20 May meeting decision):
#   * CD recorded:                  use the recorded value.
#   * CD NaN AND compl = 0:         treat as no complication (event = 0).
#   * CD NaN AND compl = 1:         severity unknown, event = NaN.
#   * CD NaN AND compl NaN:         outcome unknown, event = NaN.
#
# This replaces v10's implicit "NaN CD = no event" assumption. The event
# COUNT is the same (only CD-recorded events are confirmed), but the
# DENOMINATOR is cleaner: patients with compl = 0 contribute as confirmed
# non events, while patients with a known complication but no CD value
# are correctly flagged as unknown rather than counted as non events.
# ----------------------------------------------------------------------------

def cd_event_with_hidde_rule(cd_max_threshold):
    """Build a CD event indicator (Int64, with NaN for unknown)."""
    out = pd.Series(pd.NA, index=lung.index, dtype='Int64')
    cd_known = lung['CD_max'].notna()
    out.loc[cd_known] = (lung.loc[cd_known, 'CD_max'] >= cd_max_threshold).astype('Int64')
    inferred_no_compl = lung['CD_max'].isna() & (lung['compl'] == 0)
    out.loc[inferred_no_compl] = 0
    return out

lung['CD_ge_II']   = cd_event_with_hidde_rule(2)
lung['CD_ge_IIIa'] = cd_event_with_hidde_rule(3)
lung['CD_ge_IIIb'] = cd_event_with_hidde_rule(4)

# Report event counts and known denominators
print(f"\nDerived event indicators under Hidde's rule (full cohort, n = {len(lung):,}):")
for name in ['CD_ge_II', 'CD_ge_IIIa', 'CD_ge_IIIb']:
    s = lung[name]
    n_known = int(s.notna().sum())
    n_event = int((s == 1).sum())
    rate = 100 * n_event / n_known if n_known else 0
    print(f"  {name:<11s} events: {n_event:>4d}, known: {n_known:>4d}/{len(lung):>4d}, "
          f"rate among known: {rate:>5.1f}%")

### 7.3 Length of stay

In [ ]:
print("opnduur (Length of stay, days):")
print(f"  non null: {lung['opnduur'].notna().sum():,} ({lung['opnduur'].notna().mean()*100:.1f}%)")
print(f"  summary:")
print(lung['opnduur'].describe().to_string())

# Derived prolonged LOS flag (above the upper quartile)
p75 = lung['opnduur'].quantile(0.75)
lung['LOS_prolonged'] = lung['opnduur'] > p75
print(f"\nProlonged LOS threshold (p75): {p75:.0f} days")
print(f"  Patients with prolonged LOS: {lung['LOS_prolonged'].sum():,} "
      f"({lung['LOS_prolonged'].mean()*100:.1f}%)")


### 7.4 30 day mortality and readmission

In [ ]:
# 30 day mortality from status column
if 'status_label' in lung.columns:
    print("status (30 day vital status):")
    print(lung['status_label'].value_counts(dropna=False).to_string())

    lung['mortality_30d'] = (lung['status'] == 1).astype(int)
    n_dead = lung['mortality_30d'].sum()
    print(f"\n30 day mortality events: {n_dead:,} ({n_dead/len(lung)*100:.1f}%)")

# Readmission
if 'heropname_label' in lung.columns:
    print("\nheropname (Readmission):")
    print(lung['heropname_label'].value_counts(dropna=False).to_string())

# Cause of death distribution
print("\nCause of death (doodoorz):")
print(lung['doodoorz_label'].value_counts(dropna=False).to_string())


### 7.5 Outcome coverage by surgery year

The dictionary's Validaties sheet specifies that Clavien Dindo grade columns became mandatory only from 2022 onwards (`clavmechan` from 2024). The 8.8% overall CD coverage in Section 7.2 therefore reflects the registry's phased rollout, not a data quality problem for the cohort as a whole. The pre 2022 complication signal lives in different columns (see Section 7.6). This section reports CD coverage on the post mandate subset.

In [ ]:
print(f"CD_max coverage by surgery year subset:\n")
print(f"{'Subset':<35s} {'Patients':>10s} {'with CD_max':>14s} {'Coverage':>10s}")
print('-' * 75)

for label, mask in [
    ('Full cohort',                         pd.Series(True, index=lung.index)),
    ('surgery_year >= 2022 (CD mandate)',   lung['surgery_year'] >= 2022),
    ('surgery_year >= 2023',                lung['surgery_year'] >= 2023),
    ('surgery_year >= 2024 (clavmechan)',   lung['surgery_year'] >= 2024),
]:
    sub = lung.loc[mask]
    n = len(sub)
    if n == 0:
        continue
    n_cd = int(sub['CD_max'].notna().sum())
    coverage = n_cd / n * 100
    print(f"  {label:<33s} {n:>10,d} {n_cd:>14,d} {coverage:>9.1f}%")

# Apply the same logic to the candidate analytical cohorts for cohort A
print(f"\nWithin cohort A (Resectie longcarcinoom only), CD coverage by year:")
for label, mask in [
    ('Cohort A full',                        cohort_A),
    ('Cohort A, surgery_year >= 2022',       cohort_A & (lung['surgery_year'] >= 2022)),
]:
    n = int(mask.sum())
    if n == 0:
        continue
    n_cd = int((mask & lung['CD_max'].notna()).sum())
    coverage = n_cd / n * 100
    print(f"  {label:<35s} {n_cd:>5,d} of {n:>4,d} ({coverage:.1f}%)")

### 7.6 Pre 2022 complication signal: `postcomp2` and the legacy complication family

Before 2022 the registry did not record Clavien Dindo grades. Lung surgeons at AUMC tracked complications via a local column called `postcomp2`, and the registry also carried a family of older style per type complication flags (`complucht` for Luchtlekkage, `compneu` for Bronchopneumonie, `compinf` for Infectie, `compresp` for Respiratoir falen, `compcar` for Cardiaal incident, `comptrom` for Trombotisch incident, and others) that pre date the post 2022 `clav*` + `comp*cat` system.

This section inspects whether `postcomp2` is present in the export and reports its dtype, value distribution, and coverage by surgery year. It also reports coverage of the legacy complication family across years so we can see how the registry's recording convention shifted around 2022. The downstream linkage notebook should treat the complication outcome as a union: `clav*` derived `CD_max` for surgeries from 2022 onwards, the legacy field for surgeries before 2022.

In [ ]:
# Inspect postcomp2 if present
print("postcomp2 inspection:")
if 'postcomp2' in lung.columns:
    s = lung['postcomp2']
    print(f"  dtype:          {s.dtype}")
    print(f"  non-null:       {s.notna().sum():,} / {len(lung):,} "
          f"({s.notna().mean()*100:.1f}%)")
    print(f"\n  Value distribution:")
    print(s.value_counts(dropna=False).to_string())

    # Coverage by surgery year
    if 'surgery_year' in lung.columns:
        cov = (lung.groupby('surgery_year')
                   .agg(n=('postcomp2', 'size'),
                        n_filled=('postcomp2', lambda x: x.notna().sum())))
        cov['coverage_pct'] = (cov['n_filled'] / cov['n'] * 100).round(1)
        print(f"\n  Coverage by surgery year:")
        print(cov.to_string())
else:
    print("  postcomp2 not present in this export.")

# Inspect the legacy complication family
print("\nLegacy complication family (pre 2022 era):")
legacy_complication_cols = [
    'complucht', 'compneu', 'compinf', 'compinf2', 'compobstr', 'comphren',
    'compresp', 'compreint', 'compcar', 'comptrom', 'compover', 'compbl',
]
legacy_present = [c for c in legacy_complication_cols if c in lung.columns]
legacy_missing = [c for c in legacy_complication_cols if c not in lung.columns]

print(f"  Present in export: {len(legacy_present)} of {len(legacy_complication_cols)}")
print(f"{'column':<15s} {'dtype':<10s} {'non-null':>12s} {'positive (>0)':>15s}")
print('-' * 60)
for c in legacy_present:
    s = lung[c]
    n = int(s.notna().sum())
    n_pos = int((s > 0).sum()) if pd.api.types.is_numeric_dtype(s) else 0
    print(f"  {c:<13s} {str(s.dtype):<10s} {n:>12,d} {n_pos:>15,d}")

if legacy_missing:
    print(f"\n  Not in this export: {legacy_missing}")

# Coverage by surgery year for the legacy family combined
if legacy_present and 'surgery_year' in lung.columns:
    print(f"\n  Any legacy complication column populated, by surgery year:")
    any_legacy = lung[legacy_present].notna().any(axis=1)
    yr_cov = (lung.assign(_any=any_legacy)
                  .groupby('surgery_year')
                  .agg(n=('_any', 'size'), n_filled=('_any', 'sum')))
    yr_cov['coverage_pct'] = (yr_cov['n_filled'] / yr_cov['n'] * 100).round(1)
    print(yr_cov.to_string())

## 8. Preoperative Predictor Inventory

Identifies the candidate preoperative predictors and reports their coverage. Variables with very low coverage are flagged.

### 8.1 Demographics

In [ ]:
demographics = ['Leeftijd', 'geslacht', 'lengte', 'gewicht']

print(f"Demographic columns:")
print(f"{'column':<15s} {'non_null':>10s} {'pct_covered':>14s} {'summary'}")
print('-' * 100)
for c in demographics:
    if c in lung.columns:
        s = lung[c].dropna()
        n = len(s)
        pct = n / len(lung) * 100
        if pd.api.types.is_numeric_dtype(s):
            summary = f"mean={s.mean():.1f}, median={s.median():.1f}, range=[{s.min():.0f}, {s.max():.0f}]"
        else:
            summary = f"unique={s.nunique():,}"
        print(f"  {c:<13s} {n:>10,d} {pct:>10.1f}%    {summary}")

# Derive BMI
if 'lengte' in lung.columns and 'gewicht' in lung.columns:
    lung['BMI'] = lung['gewicht'] / (lung['lengte'] / 100) ** 2
    n_bmi = lung['BMI'].notna().sum()
    print(f"\nDerived BMI:")
    print(f"  computable: {n_bmi:,} ({n_bmi/len(lung)*100:.1f}%)")
    if n_bmi > 0:
        print(f"  summary: mean={lung['BMI'].mean():.1f}, median={lung['BMI'].median():.1f}, "
              f"range=[{lung['BMI'].min():.1f}, {lung['BMI'].max():.1f}]")


### 8.2 Performance status and fitness

In [ ]:
for c, label_col in [('ecog', 'ecog_label'), ('asascore', 'asascore_label'), ('Stage', None)]:
    if c in lung.columns:
        s = lung[c].dropna()
        n = len(s)
        pct = n / len(lung) * 100
        print(f"\n{c} (coverage: {n:,}, {pct:.1f}%)")
        if label_col and label_col in lung.columns:
            print(lung[label_col].value_counts(dropna=False).to_string())
        else:
            print(s.value_counts().sort_index().to_string())


### 8.3 Pulmonary function

After recoding 999 and 99.9 sentinel values to `NaN` in Section 4, the true coverage of FEV1 %, DLCO %, and VO2 max is reported here.

In [ ]:
pulm_cols = ['prelongf', 'prelongf1', 'prelongfvo2max']

print(f"Pulmonary function columns (after sentinel recoding):")
print(f"{'column':<20s} {'non_null':>10s} {'pct_covered':>14s} {'summary'}")
print('-' * 100)
for c in pulm_cols:
    if c in lung.columns:
        s = lung[c].dropna()
        n = len(s)
        pct = n / len(lung) * 100
        if n > 0:
            summary = f"mean={s.mean():.1f}, median={s.median():.1f}, range=[{s.min():.1f}, {s.max():.1f}]"
        else:
            summary = "(empty)"
        print(f"  {c:<18s} {n:>10,d} {pct:>10.1f}%    {summary}")


### 8.4 Smoking status

In [ ]:
if 'roken_label' in lung.columns:
    s = lung['roken'].dropna()
    print(f"roken (Smoking status):")
    print(f"  non null: {len(s):,} ({len(s)/len(lung)*100:.1f}%)")
    print(f"  value counts:")
    print(lung['roken_label'].value_counts(dropna=False).to_string())
    print(f"\nWith 95% missingness, smoking cannot reliably be used as an individual patient predictor.")
    print(f"This is flagged in Section 10 as an open question for Edwin.")


### 8.5 Comorbidities

In [ ]:
# Comorbidity flag columns (prefix com but not comp which is complications).
# v6: also exclude _label twin columns — they have the same low-cardinality
# pattern as the binary flags themselves and would otherwise sneak in.
comorbidity_cols = [c for c in lung.columns
                    if c.startswith('com')
                    and not c.startswith('comp')
                    and not c.endswith('_label')
                    and lung[c].nunique(dropna=True) <= 5]

print(f"Comorbidity flag columns ({len(comorbidity_cols)}):\n")
print(f"{'column':<20s} {'non_null':>10s} {'positive_count':>16s}")
print('-' * 55)
for c in comorbidity_cols:
    s = lung[c].dropna()
    n = len(s)
    n_pos = int((s > 0).sum()) if pd.api.types.is_numeric_dtype(s) else 0
    print(f"  {c:<18s} {n:>10,d} {n_pos:>16,d}")

# Derive comorbidity count per patient (approximates Charlson)
lung['n_comorbidities'] = lung[comorbidity_cols].apply(
    lambda r: (r.fillna(0) > 0).sum(), axis=1
)
print(f"\nDerived comorbidity count per patient:")
print(lung['n_comorbidities'].describe().to_string())

### 8.6 Predictor coverage by mandatory by date applicability

Several predictors are only mandatory from specific surgery years onwards. Reporting coverage on the full cohort understates the true completeness. This section reports coverage on the applicable subset for each predictor and runs the dictionary's declared range warnings.

In [ ]:
# Pulmonary function: mandatory for reggroep=1 from 2017,
# with the rule that prelongf1 OR prelongfvo2max must be filled
print("Pulmonary function (mandatory for reggroep=1 from 2017):")
applicable = (lung['reggroep'] == 1) & (lung['surgery_year'] >= 2017)
sub = lung.loc[applicable]
print(f"  Applicable subset size: {len(sub):,}\n")

for c in ['prelongf', 'prelongf1', 'prelongfvo2max']:
    if c in sub.columns:
        n = int(sub[c].notna().sum())
        print(f"    {c:<20s} {n:>5,d} of {len(sub):,} ({n/len(sub)*100:.1f}%)")

# Dictionary rule: at least one of prelongf1 or prelongfvo2max must be filled
either_filled = sub['prelongf1'].notna() | sub['prelongfvo2max'].notna()
print(f"    DLCO or VO2 max either filled: "
      f"{int(either_filled.sum()):,} of {len(sub):,} "
      f"({either_filled.mean()*100:.1f}%)")

# Demographics and ECOG: mandatory from 2023
print(f"\nDemographics and ECOG (mandatory from 2023):")
applicable = lung['surgery_year'] >= 2023
sub = lung.loc[applicable]
print(f"  Applicable subset size: {len(sub):,}\n")
for c in ['lengte', 'gewicht', 'ecog']:
    if c in sub.columns:
        n = int(sub[c].notna().sum())
        print(f"    {c:<20s} {n:>5,d} of {len(sub):,} ({n/len(sub)*100:.1f}%)")

# Range checks from the Validaties sheet
print(f"\nRange checks (rules from DLSA Dictionary):")
range_rules = [
    ('lengte',    100, 230, 'warning'),
    ('gewicht',    25, 350, 'warning'),
    ('prelongf',    0, 150, 'error'),
    ('prelongf1',   0, 150, 'error'),
]
for v, lo, hi, level in range_rules:
    if v not in lung.columns:
        continue
    s = lung[v].dropna()
    if len(s) == 0:
        continue
    bad = int(((s < lo) | (s > hi)).sum())
    flag = "OK" if bad == 0 else "ATTENTION"
    print(f"  {v:<15s} expected [{lo}, {hi}] ({level}): "
          f"{bad:,} values outside  {flag}")

### 7.7 Note on CD and Length of Stay Correlation

CD severity and prolonged length of stay are clinically correlated. A patient with a high postoperative stay almost always had more complications and more intensive treatment. This is an **association**, not a causal mechanism. The two outcomes are not independent.

When reporting the analysis, frame LOS and CD together with explicit acknowledgement of this correlation. If we model LOS as a separate outcome, the discussion notes that it captures complication severity indirectly through resource utilisation rather than as a clean independent endpoint.

This was agreed at the 20 May meeting with Hidde and Marieke.

## 9. Primary Outcome Decision Table

Consolidates the candidate primary outcomes into a single table showing cohort size, event count, event rate, and predictor budget (events / 10) per outcome and per cohort. The table is saved to the project outputs folder for the Edwin meeting.

In [ ]:
# Build the outcome decision table.
# Under Hidde's rule (20 May meeting), CD-based outcomes use the new event
# indicators that distinguish confirmed non events (compl = 0) from unknown
# outcomes (compl missing or compl = 1 without CD). The new n_known column
# shows the denominator of patients with a confirmed outcome.

def make_outcome_row(name, event_series, cohort_mask, event_type='binary'):
    n_cohort = int(cohort_mask.sum())
    if event_type == 'cd_int64':
        # Int64 series with NaN for unknown
        s = event_series[cohort_mask]
        n_known = int(s.notna().sum())
        n_event = int((s == 1).sum())
    else:
        # Plain boolean series, NaN treated as False (e.g. compl, mortality, LOS)
        aligned = event_series.fillna(False) if event_series.dtype != bool else event_series
        n_known = n_cohort
        n_event = int((cohort_mask & aligned).sum())
    rate = n_event / n_known * 100 if n_known else 0
    budget = n_event // 10
    return {
        'outcome': name,
        'cohort_size': n_cohort,
        'n_known': n_known,
        'events': n_event,
        'event_rate_pct': round(rate, 1),
        'predictor_budget': budget,
    }

candidates = [
    ('compl = Ja (any complication)', lung['compl'] == 1,         'binary'),
    ('CD_max >= II',                  lung['CD_ge_II'],            'cd_int64'),
    ('CD_max >= IIIa',                lung['CD_ge_IIIa'],          'cd_int64'),
    ('CD_max >= IIIb (Hidde major)',  lung['CD_ge_IIIb'],          'cd_int64'),
    ('Prolonged LOS (> p75)',         lung['LOS_prolonged'],       'binary'),
    ('30 day mortality',              lung['status'] == 1,         'binary'),
]

cohorts = {
    'A: Resectie longcarcinoom':              cohort_A,
    'B: Resectie + Metastasectomie':          cohort_B,
    'C: All thoracic surgery':                cohort_C,
}

rows = []
for cohort_name, mask in cohorts.items():
    for outcome_name, series, event_type in candidates:
        row = make_outcome_row(outcome_name, series, mask, event_type)
        row['cohort'] = cohort_name
        rows.append(row)

decision_table = pd.DataFrame(rows)[['cohort', 'outcome', 'cohort_size', 'n_known',
                                     'events', 'event_rate_pct', 'predictor_budget']]
print("Primary outcome decision table (Hidde's rule applied to CD outcomes):")
print(decision_table.to_string(index=False))

In [ ]:
# Save the decision table for the Edwin meeting
output_dir = Path(r"Z:\Users\predicting recovery\AmanDeep\Data Analysis\lung_analysis\output_files")
output_dir.mkdir(parents=True, exist_ok=True)

decision_path = output_dir / "lung_primary_outcome_decision_table.csv"
decision_table.to_csv(decision_path, index=False, encoding='utf-8')
print(f"\nOutcome decision table saved to: {decision_path}")


## 10. Open Questions for Edwin and Reference Decisions

### 10.1 Cohort definition

Three candidate cohorts in Section 6: primary lung cancer resection only (A), plus metastasectomy (B), or all thoracic surgery (C). Section 9 gives event rates and predictor budgets per cohort.

### 10.2 Primary outcome and clinical thresholds

Six candidate outcomes in Section 9: any complication, CD `>=` II, CD `>=` IIIa, CD `>=` IIIb, prolonged length of stay, 30 day mortality. Each has a different event rate and predictor budget under Hidde's rule.

**Decision from 20 May meeting (Hidde):** CD `>=` IIIb is the agreed clinical threshold for a major postoperative complication (a complication requiring intervention under anaesthesia). It is the most clinically defensible severity outcome. However, on Cohort A the event count is low (22 events on the recorded subset), which constrains the predictor budget for multivariable modelling. The `compl = 1` outcome remains the practical primary outcome for lung modelling given event count, with CD `>=` IIIb available for sensitivity analysis.

### 10.3 Smoking status coverage

Coverage is 5.0% (60 of 1,200 patients). The dictionary's Validaties sheet declares no date based mandate for `roken`, so this is not a phased rollout artefact. Smoking is persistently under recorded across 2012 to 2025. Three options: accept the small subset for a sensitivity analysis, leave smoking out of the model, or ask the data team to recover historical smoking from the EHR or physiotherapy assessments.

### 10.4 UPN outliers and duplicates

Six UPN values do not normalise to 7 digits (in `lung_upn_outliers.csv`). Duplicate UPNs from Section 3.4 are now deduplicated per Section 4.7.2 (Hidde's rule, keep first surgery per UPN). The eight duplicated UPN cases (16 rows) have been reduced to eight rows, with the second surgery dropped as agreed.

### 10.5 Metadata drift, resolved by the dictionary

Six variables (`compinfect`, `compluchtweg`, `comprespi`, `ecog`, `heropname`, `roken`) have value labels in the dictionary but not in the `.sav`. Section 2.4.1 confirms the fallback maps match the dictionary exactly. Worth flagging to the data team so future `.sav` exports carry the option set declarations.

### 10.6 Coverage interpretation per dictionary mandates

Section 2.4.2 lists the mandatory by date thresholds. Sections 7.5 and 8.6 report coverage on the applicable subsets. Low overall coverage on `clav*`, `lengte`, `gewicht`, and `ecog` is a feature of the registry's phased rollout, not a data quality issue.

### 10.7 Date inconsistencies flagged by Section 4.6

Three rows had `datok` on or before `gebdat` (the implausibly recent birthdates). These were excluded by the age `<` 18 rule in Section 4.7.1 per Hidde's decision. 99 rows have `datont` before `datok` (discharge before surgery, 8% of cohort); the registry's `opnduur` field versus the computed value needs Edwin's confirmation as the source of truth. One row has `status` = Overleden but `datovl` null; minor data quality flag.

### 10.8 Reference: data quality decisions from the 20 May meeting

Applied in Section 4.7 of this notebook.

* **Age outliers:** Exclude patients with age outside 18 to 100.
* **Duplicate MDNs:** Keep the first surgery per patient as the index event.
* **NaN Clavien Dindo:** Where CD is missing and `compl = 0`, treat as no complication. Where CD is missing and `compl = 1` or NaN, leave as unknown.
* **Major complication threshold:** CD `>=` IIIb (under anaesthesia intervention).
* **Imputation strategy:** Primary analysis on complete cases. Sensitivity with selective imputation on clinically interesting predictors only. Applied in the downstream modelling notebook, not here.

### 10.9 What changes for Edwin's review

The cohort sizes in Section 6 and 9 reflect the new exclusions. The outcome event counts in the decision table now use Hidde's rule for CD outcomes. Edwin's role at the 11:00 meeting is to confirm cohort definition (A vs B vs C) and primary outcome (compl vs CD `>=` IIIb).

## 11. Save the Prepared Dataset

We save the cleaned and labelled DataFrame to the project outputs folder as a CSV. The `.sav` metadata is not preserved in the CSV roundtrip; the labels are already materialised as `_label` columns and the sentinel recoded values are now proper `NaN`. If the SPSS metadata is needed later we can re read the `.sav`.

In [ ]:
# Save the prepared dataset
output_path = output_dir / "lung_prepared.csv"
lung.to_csv(output_path, index=False, encoding='utf-8')
print(f"Prepared lung dataset saved to: {output_path}")
print(f"  Rows: {len(lung):,}")
print(f"  Columns: {lung.shape[1]:,} (includes derived _label, _dt, and event indicator columns)")


## 12. Linkage Cohort Size Check

Before building the downstream linkage and modelling notebook, we check how many patients in the lung surgical cohort overlap with the two physiotherapy assessment outputs from `data-exploration-v11.ipynb`. The intersection sizes determine the analytical sample size for RQ1 and RQ2 modelling on the lung cohort.

* `assessment_conclusie_edwin_dedup.csv` (598 rows, 586 patients) carries the 17 ICF classifier features and is the predictor source for RQ2.
* `edwin_geleijn_assessments_parsed.csv` (543 rows) carries the 12 structured fields from the 2007 risk scoring form and is the predictor source for RQ1.

In [ ]:
# 12. Linkage cohort size check
edwin_dir = Path(r"Z:\Users\predicting recovery\AmanDeep\Data Analysis\edwin_notes_ICF_output_analysis\output_files")

edwin_dedup = pd.read_csv(edwin_dir / "assessment_conclusie_edwin_dedup.csv", dtype={'MDN': str})

# Defensive: pad MDN to 7 digit string just like the lung UPN
edwin_dedup['MDN'] = edwin_dedup['MDN'].astype(str).str.zfill(7)

lung_upns   = set(lung['UPN'].dropna().astype(str))
edwin_mdns  = set(edwin_dedup['MDN'].dropna())

print("Cohort and assessment counts:")
print(f"  Lung surgical cohort:              {len(lung_upns):,}")
print(f"  Edwin dedup patients (RQ2 source): {len(edwin_mdns):,}")

intersect_dedup  = lung_upns & edwin_mdns

print("\nLinkage intersections:")
print(f"  Lung intersected with Edwin dedup: {len(intersect_dedup):,}  (RQ2 candidate cohort)")

# Outcome event counts in the RQ2 candidate cohort
if intersect_dedup:
    linked = lung[lung['UPN'].isin(intersect_dedup)]
    compl_numeric = pd.to_numeric(linked['compl'], errors='coerce')
    n_any = int((compl_numeric == 1).sum())
    n_any_known = int(compl_numeric.notna().sum())
    n_cd_ge_ii = int((linked['CD_max'] >= 2).sum()) if 'CD_max' in linked.columns else None
    n_cd_ge_iiia = int((linked['CD_max'] >= 3).sum()) if 'CD_max' in linked.columns else None

    print("\nOutcome events in lung intersected with Edwin dedup:")
    print(f"  any complication (compl = 1):    {n_any} of {n_any_known} known "
          f"({100*n_any/max(n_any_known,1):.1f}%)")
    if n_cd_ge_ii is not None:
        print(f"  CD_max >= II (any CD recorded):                  {n_cd_ge_ii}")
        print(f"  CD_max >= IIIa:                                  {n_cd_ge_iiia}")

# Save the intersected UPN list for the downstream linkage notebook to consume
output_dir = Path(r"Z:\Users\predicting recovery\AmanDeep\Data Analysis\lung_analysis\output_files")
output_dir.mkdir(parents=True, exist_ok=True)
intersection_path = output_dir / "lung_edwin_intersection_upns.csv"
pd.DataFrame({
    'UPN': sorted(intersect_dedup)}).to_csv(intersection_path, index=False)
print(f"\nSaved intersection UPNs to: {intersection_path}")


## Notebook Complete

**Files saved to `Z:\Users\predicting recovery\AmanDeep\Data Analysis\lung_analysis\output_files\`:**

* `lung_upn_outliers.csv`. UPN values that did not normalise to 7 digits.
* `lung_upn_duplicates.csv`. UPN values appearing on more than one row.
* `lung_primary_outcome_decision_table.csv`. Candidate outcomes by candidate cohorts with event counts, rates, and predictor budgets.
* `lung_prepared.csv`. Full prepared DataFrame with UPN normalised, sentinels recoded to `NaN`, dates parsed, value labels applied, and derived columns (`BMI`, `CD_max`, `CD_ge_II`, `CD_ge_IIIa`, `CD_ge_IIIb`, `LOS_prolonged`, `mortality_30d`, `n_comorbidities`, `surgery_year`).

**In memory:**

* `lung`. The full prepared DataFrame.
* `meta`. The original SPSS metadata object from `pyreadstat`.
* `dict_vars`, `dict_opties`, `dict_validations`, `mandate_by_date`. DLSA Dictionary structures (only populated if the dictionary loaded).
* `LABEL_MAPS`, `CD_GRADE_MAP`, `CLAV_COLS`, `cohort_A`, `cohort_B`, `cohort_C`. Supporting structures.

### Linkage notebook handoff notes

* UPN is a 7 digit string after normalisation, ready to inner join against MDN in the Edwin notes outputs from `data-exploration-v11`.
* `CD_max` and the `CD_ge_*` event indicators are pre computed.
* The outliers and duplicates rows are still present in `lung_prepared.csv`. Edwin's decisions on pad versus exclude and on whether to keep the multi surgery rows need to be applied at the linkage step.

### Next analytical steps

1. Surgery date anchored linkage to the Edwin notes assessment cohort. Inner join on UPN to MDN, filter notes by `NOTITIE_DATUM_dt < datok_dt`.
2. Replicate this notebook's structure for the upper GI cohort.
3. Once Edwin confirms primary outcome, fit explainable RQ1 models on the linked cohort.